# Virtual Screening for Novel MMP-9 Inhibitors

In this notebook, we deploy our trained Random Forest champion model to perform **Ligand-Based Virtual Screening (LBVS)**. 
We will take a library of unlabelled, novel chemical compounds, convert them into molecular fingerprints, and use our model to predict which compounds are most likely to be active MMP-9 inhibitors.

In [3]:
import pandas as pd
import numpy as np
import joblib as joblib
import os

from rdkit import Chem
from rdkit.Chem import AllChem

# Suppress RDKit warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

print("Virtual Screening environment ready!")

Virtual Screening environment ready!


## 1. Load the Champion Model and Metadata
We load the serialized Random Forest model and the metadata dictionary we saved at the end of the training phase. The metadata ensures we use the exact same fingerprint parameters and decision threshold that were optimized during training.

In [4]:
# Define paths
model_path = '../models/mmp9_rf_champion.pkl'
meta_path = '../models/mmp9_rf_metadata.pkl'

print("Loading model and metadata...")

# Load the model and metadata
try:
    rf_champ = joblib.load(model_path)
    model_metadata = joblib.load(meta_path)
    
    # Extract deployment parameters
    optimal_threshold = model_metadata['optimal_threshold']
    fp_radius = model_metadata['fingerprint_radius']
    fp_bits = model_metadata['n_features']
    
    print("--- Deployment Parameters Loaded ---")
    print(f"Model Type: {model_metadata['model_type']}")
    print(f"Optimal Decision Threshold: {optimal_threshold:.4f}")
    print(f"Expected Fingerprint Radius: {fp_radius}")
    print(f"Expected Fingerprint Bits: {fp_bits}")
    
except FileNotFoundError:
    print("Error: Could not find the model files. Ensure the training notebook saved them correctly.")

Loading model and metadata...
--- Deployment Parameters Loaded ---
Model Type: RandomForestClassifier
Optimal Decision Threshold: 0.4417
Expected Fingerprint Radius: 2
Expected Fingerprint Bits: 2048


## 2. Acquire a Chemical Library (ChEMBL)
Instead of screening billions of random molecules, we will perform a **Drug Repurposing** screen. We will use the ChEMBL API to download a library of currently approved drugs (Phase 4). Our goal is to see if any existing, safe medications have hidden MMP-9 inhibitory properties.

In [ ]:
# ChEMBL client to fetch bioactivity data
from chembl_webresource_client.new_client import new_client
print("Connecting to ChEMBL Database...")

# Access the molecule endpoint
molecule = new_client.molecule

# Filter for approved drugs (max_phase = 4 means fully approved)
print("Querying for approved drugs. This might take a minute or two...")
approved_drugs = molecule.filter(max_phase=4)

# Extract the ChEMBL ID, preferred name, and canonical SMILES
drug_list = []
for drug in approved_drugs:
    # Make sure the drug actually has a SMILES string (some large biologics don't)
    if drug.get('molecule_structures') and drug['molecule_structures'].get('canonical_smiles'):
        drug_list.append({
            'chembl_id': drug['molecule_chembl_id'],
            'name': drug.get('pref_name', 'Unknown'),
            'smiles': drug['molecule_structures']['canonical_smiles']
        })

# Convert to a pandas DataFrame
df_screen = pd.DataFrame(drug_list)

# Drop any duplicates just in case
df_screen = df_screen.drop_duplicates(subset='smiles').reset_index(drop=True)

print(f"Success! Downloaded {len(df_screen)} approved drugs with valid SMILES strings.")
display(df_screen.head())

Connecting to ChEMBL Database...
Querying for approved drugs. This might take a minute or two...
Success! Downloaded 3229 approved drugs with valid SMILES strings.


,chembl_id,name,smiles
0,CHEMBL2,PRAZOSIN,COc1cc2nc(N3CCN(C(=O)c4ccco4)CC3)nc(N)c2cc1OC
1,CHEMBL3,NICOTINE,CN1CCC[C@H]1c1cccnc1
2,CHEMBL4,OFLOXACIN,CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23
3,CHEMBL5,NALIDIXIC ACID,CCn1cc(C(=O)O)c(=O)c2ccc(C)nc21
4,CHEMBL6,INDOMETHACIN,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1


## 3. ADMET Pre-Filtering (Lipinski's Rule of 5)
To ensure our predicted compounds make good oral drug candidates, we filter our library using Lipinski's Rules:
* Molecular Weight $\le$ 500 Daltons
* LogP (Lipophilicity) $\le$ 5
* Hydrogen Bond Donors $\le$ 5
* Hydrogen Bond Acceptors $\le$ 10

In [6]:
from rdkit.Chem import Descriptors

print("Applying Lipinski's Rule of 5...")

def check_lipinski(smiles):
    """Evaluates if a SMILES string strictly passes Lipinski's Rule of 5."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False
        
        # Calculate the 4 Lipinski parameters
        mw = Descriptors.ExactMolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        
        # Strict evaluation (must pass all 4)
        if (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10):
            return True
        return False
    except:
        return False # Drop molecule if RDKit can't parse it

# Apply the filter to our dataframe
df_screen['Passes_Lipinski'] = df_screen['smiles'].apply(check_lipinski)

# Keep only the drugs that passed
df_filtered = df_screen[df_screen['Passes_Lipinski'] == True].copy().reset_index(drop=True)

drugs_dropped = len(df_screen) - len(df_filtered)
print(f"Dropped {drugs_dropped} heavy/greasy drugs.")
print(f"Library reduced to {len(df_filtered)} drug-like candidates for the ML model.")

Applying Lipinski's Rule of 5...


[15:53:31] WARNING: not removing hydrogen atom without neighbors
[15:53:31] WARNING: not removing hydrogen atom without neighbors
[15:53:31] WARNING: not removing hydrogen atom without neighbors
[15:53:31] WARNING: not removing hydrogen atom without neighbors


Dropped 1085 heavy/greasy drugs.
Library reduced to 2144 drug-like candidates for the ML model.


## 4. Virtual Screening (ML Interrogation)
We convert our filtered, drug-like library into Morgan Fingerprints and feed them into our optimized Random Forest model. We will use the custom probability threshold calculated during training to isolate the highest-confidence hits.

In [7]:
def get_morgan_fingerprint(smiles, radius=2, n_bits=2048):
    """Generate Morgan Fingerprint for ML input."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros((n_bits,))
        return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits))
    except:
        return np.zeros((n_bits,))

print("Translating SMILES to Morgan Fingerprints...")
# Calculate fingerprints using the parameters saved in our metadata
fp_list = df_filtered['smiles'].apply(lambda x: get_morgan_fingerprint(x, radius=fp_radius, n_bits=fp_bits))

# Stack into a 2D matrix
X_screen = np.stack(fp_list.values)

print("Running compounds through the Champion Random Forest model...")
# Predict the probability of each drug inhibiting MMP-9
df_filtered['Active_Probability'] = rf_champ.predict_proba(X_screen)[:, 1]

# Apply our mathematically optimized threshold (0.4417)
df_filtered['Predicted_Active'] = (df_filtered['Active_Probability'] >= optimal_threshold).astype(int)

# Count our hits
total_hits = df_filtered['Predicted_Active'].sum()
print(f"Screening Complete! The model flagged {total_hits} potential MMP-9 inhibitors.")

Translating SMILES to Morgan Fingerprints...


[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerator
[15:55:09] DEPRECATION WARNING: please use MorganGenerat

Running compounds through the Champion Random Forest model...
Screening Complete! The model flagged 558 potential MMP-9 inhibitors.


## 5. Review the Hits (The Triage)
We have successfully screened thousands of approved drugs. Now, we filter the dataset to only show the compounds that passed our optimized threshold (0.4417). We sort these "hits" by the model's prediction probability to give the wet-lab researchers our absolute highest-confidence recommendations.

In [9]:
# Isolate the hits (Predicted_Active == 1)
hits_df = df_filtered[df_filtered['Predicted_Active'] == 1].copy()

# Sort by the highest probability (Confidence)
hits_df = hits_df.sort_values(by='Active_Probability', ascending=False).reset_index(drop=True)

print("🏆 Top 20 Drug Repurposing Candidates for MMP-9 Inhibition 🏆")
# Display the most important columns for the researchers
display(hits_df[['chembl_id', 'name', 'Active_Probability', 'smiles']].head(20))

# Save the successful hits to a CSV for the "Pharmacist"
output_hits_file = '../data/processed/virtual_screening_hits.csv'
hits_df.to_csv(output_hits_file, index=False)
print(f"\nSaved all {len(hits_df)} hits to {output_hits_file}")

🏆 Top 20 Drug Repurposing Candidates for MMP-9 Inhibition 🏆


,chembl_id,name,Active_Probability,smiles
0,CHEMBL1873475,IBRUTINIB,0.704232,C=CC(=O)N1CCC[C@@H](n2nc(-c3ccc(Oc4ccccc4)cc3)...
1,CHEMBL2106155,ENCAINIDE HYDROCHLORIDE,0.686235,COc1ccc(C(=O)Nc2ccccc2CCC2CCCCN2C)cc1.Cl
2,CHEMBL315838,ENCAINIDE,0.686235,COc1ccc(C(=O)Nc2ccccc2CCC2CCCCN2C)cc1
3,CHEMBL1200788,CISAPRIDE MONOHYDRATE,0.679120,COc1cc(N)c(Cl)cc1C(=O)NC1CCN(CCCOc2ccc(F)cc2)C...
4,CHEMBL1729,CISAPRIDE,0.676731,COc1cc(N)c(Cl)cc1C(=O)NC1CCN(CCCOc2ccc(F)cc2)C...
5,CHEMBL3936761,ZANUBRUTINIB,0.676428,C=CC(=O)N1CCC([C@@H]2CCNc3c(C(N)=O)c(-c4ccc(Oc...
6,CHEMBL1200478,DYCLONINE HYDROCHLORIDE,0.674737,CCCCOc1ccc(C(=O)CCN2CCCCC2)cc1.Cl
7,CHEMBL1201217,DYCLONINE,0.674737,CCCCOc1ccc(C(=O)CCN2CCCCC2)cc1
8,CHEMBL5314600,TIRABRUTINIB HYDROCHLORIDE,0.664978,CC#CC(=O)N1CC[C@@H](n2c(=O)n(-c3ccc(Oc4ccccc4)...
9,CHEMBL4071161,TIRABRUTINIB,0.664978,CC#CC(=O)N1CC[C@@H](n2c(=O)n(-c3ccc(Oc4ccccc4)...



Saved all 558 hits to ../data/processed/virtual_screening_hits.csv
